In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Load vector store
chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_collection("layoffs_rag")

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Vector store loaded: {collection.count()} documents")
print("OpenAI client ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store loaded: 2782 documents
OpenAI client ready


In [3]:
def retrieve_context(query, n_results=5, source_filter=None):
    """
    Retrieve relevant chunks from the vector store for a given query.
    source_filter: 'layoffs', 'news', or None for both
    """
    where_filter = {"source": source_filter} if source_filter else None
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where_filter
    )
    
    chunks = results['documents'][0]
    metadatas = results['metadatas'][0]
    
    return chunks, metadatas

# Test it
chunks, metadatas = retrieve_context("Amazon layoffs 2023")
print(f"Retrieved {len(chunks)} chunks:\n")
for i, (chunk, meta) in enumerate(zip(chunks, metadatas)):
    print(f"{i+1}. [{meta['source']}] {chunk[:120]}...")
    print()

Retrieved 5 chunks:

1. [layoffs] Amazon, based in Seattle (United States), laid off an unknown number of employees  on January 29, 2025. They operate in ...

2. [layoffs] Amazon, based in Seattle (United States), laid off an unknown number of employees  on July 17, 2025. They operate in the...

3. [layoffs] Amazon, based in Seattle (United States), laid off 180 employees  on November 13, 2023. They operate in the Retail indus...

4. [layoffs] Amazon, based in Seattle (United States), laid off an unknown number of employees  on November 17, 2023. They operate in...

5. [layoffs] Amazon, based in Seattle (United States), laid off 14000 employees (1% of workforce) on October 27, 2025. They operate i...



In [4]:
def generate_answer(query, chunks):
    """
    Generate an answer using retrieved chunks as context.
    """
    # Format chunks into a single context string
    context = "\n\n".join([f"Source {i+1}: {chunk}" for i, chunk in enumerate(chunks)])
    
    # Build the prompt
    prompt = f"""You are an expert analyst of tech industry layoff trends. 
Answer the question using ONLY the provided context. 
If the context doesn't contain enough information to answer fully, say so honestly.
Always mention specific companies, numbers, and dates when available in the context.

Context:
{context}

Question: {query}

Answer:"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful tech industry analyst. Answer questions based only on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1
    )
    
    return response.choices[0].message.content

# Test it
answer = generate_answer("Amazon layoffs 2023", chunks)
print(answer)

In 2023, Amazon laid off 180 employees on November 13. Additionally, they laid off an unknown number of employees on November 17.


In [5]:
def ask_layoffs_assistant(query, n_results=5, source_filter=None):
    """
    Full RAG pipeline: retrieve relevant context then generate answer.
    """
    # Step 1: Retrieve
    chunks, metadatas = retrieve_context(query, n_results, source_filter)
    
    # Step 2: Generate
    answer = generate_answer(query, chunks)
    
    return {
        "query": query,
        "answer": answer,
        "sources": metadatas,
        "chunks_used": chunks
    }

# Test with multiple questions
questions = [
    "Which industry had the most layoffs overall?",
    "What was the media sentiment during the 2023 tech layoff wave?",
    "How did AI companies compare to non-AI companies in layoffs?"
]

for q in questions:
    print(f"Q: {q}")
    result = ask_layoffs_assistant(q)
    print(f"A: {result['answer']}")
    print()

Q: Which industry had the most layoffs overall?
A: The industry with the most layoffs overall is the "Other" category, with a total of 86,943 employees laid off.

Q: What was the media sentiment during the 2023 tech layoff wave?
A: The media sentiment during the 2023 tech layoff wave was predominantly negative, with 150 out of 306 analyzed articles being negative, 119 positive, and 37 neutral. The average sentiment score was -0.0397.

Q: How did AI companies compare to non-AI companies in layoffs?
A: AI companies laid off a total of 149,030 employees, while non-AI companies laid off 392,263 employees. This indicates that non-AI companies had a significantly higher number of layoffs compared to AI companies.



In [6]:
result = ask_layoffs_assistant("Which industry had the most layoffs overall?", n_results=20)
print(result['answer'])

The industry with the most layoffs overall is the "Other" industry, with a total of 86,943 employees laid off.


In [7]:
result = ask_layoffs_assistant("What was the media sentiment during the 2023 tech layoff wave?")
print(result['answer'])

The media sentiment during the 2023 tech layoff wave was predominantly negative, with 150 out of 306 analyzed articles being negative, 119 positive, and 37 neutral. The average sentiment score during this period was -0.0397.
